# AF Entropy Analysis — Stepwise (verified against hand-worked derivation)

This notebook is structured in the same step order as the manual derivation:
1. **Part 1** — Approximate Entropy (ApEn), worked step-by-step on a small test series `x = [4,7,9,10,6]`
2. **Part 2** — Sample Entropy (SampEn), same test series
3. **Part 3** — Wrap the verified logic into reusable functions and confirm they reproduce the same numbers
4. **Part 4** — Normalization of AF(d) (Section 3.3 of the paper: d/R and AF/total)
5. **Part 5** — Apply everything to the real Ganges `Chennel_Data.xlsx` data, year by year

Every intermediate table (distance matrix, matching counts, phi values) is printed so you can check it against your notebook page.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
# PART 1 — Approximate Entropy (manual, worked step-by-step)

### Step 1 — Time series, N, m, n(=m+1)

In [ ]:
x = np.array([4, 7, 9, 10, 6])
N = len(x)
m = 2
n = m + 1   # standard parameter, i.e. m+1
r = 0.15*np.std(x)       # tolerance (fixed value used here to match the worked example)

print("Time series x =", x)
print("N (number of data points) =", N)
print("m =", m, " n (m+1) =", n)
print("r (tolerance) =", r)

Time series x = [ 4  7  9 10  6]
N (number of data points) = 5
m = 2  n (m+1) = 3
r (tolerance) = 3


### Step 2 — Form vectors of length m = 2

Number of vectors = N - m + 1 = 4

In [10]:
def form_vectors(series, length):
    N = len(series)
    return np.array([series[i:i + length] for i in range(N - length + 1)])

vecs_m = form_vectors(x, m)

vec_table = pd.DataFrame({
    "Vector": [f"x{i+1}" for i in range(len(vecs_m))],
    "Values": [list(v) for v in vecs_m]
})
print(f"Number of vectors = N - m + 1 = {len(vecs_m)}")
vec_table

Number of vectors = N - m + 1 = 4


,Vector,Values
0,x1,"[4, 7]"
1,x2,"[7, 9]"
2,x3,"[9, 10]"
3,x4,"[10, 6]"


### Step 3 — Distance matrix (Chebyshev / max-abs-difference) between all length-m vectors

`d(xi, xj) = max(|a1 - b1|, |a2 - b2|)`

In [11]:
def distance_matrix(vecs):
    n_vec = len(vecs)
    D = np.zeros((n_vec, n_vec))
    for i in range(n_vec):
        for j in range(n_vec):
            D[i, j] = np.max(np.abs(vecs[i] - vecs[j]))
    return D

D_m = distance_matrix(vecs_m)

labels_m = [f"x{i+1}" for i in range(len(vecs_m))]
D_m_df = pd.DataFrame(D_m, index=labels_m, columns=labels_m)
D_m_df

,x1,x2,x3,x4
x1,0.0,3.0,5.0,6.0
x2,3.0,0.0,2.0,3.0
x3,5.0,2.0,0.0,4.0
x4,6.0,3.0,4.0,0.0


### Step 4 — Matching count and Ci for each vector (ApEn INCLUDES self-match, i.e. the diagonal, since d(xi,xi)=0 ≤ r)

In [12]:
def compute_Ci(D, r):
    n_vec = D.shape[0]
    C = np.sum(D <= r, axis=1) / n_vec   # includes self-match (diagonal = 0 <= r)
    return C

C_m = compute_Ci(D_m, r)

C_m_df = pd.DataFrame({
    "Vector": labels_m,
    "Matches (d<=r, incl. self)": np.sum(D_m <= r, axis=1),
    "Ci": C_m
})
C_m_df

,Vector,"Matches (d<=r, incl. self)",Ci
0,x1,2,0.5
1,x2,4,1.0
2,x3,2,0.5
3,x4,2,0.5


### Step 5 — phi^m = mean( ln(Ci) )

In [13]:
phi_m = np.mean(np.log(C_m))
print(f"phi^m = {phi_m:.4f}")

phi^m = -0.5199


### Step 6 — Repeat Steps 2–5 for vector length m+1 = 3

In [14]:
vecs_n = form_vectors(x, n)
labels_n = [f"Y{i+1}" for i in range(len(vecs_n))]

vec_table_n = pd.DataFrame({
    "Vector": labels_n,
    "Values": [list(v) for v in vecs_n]
})
print(f"Number of vectors = N - (m+1) + 1 = {len(vecs_n)}")
display(vec_table_n)

D_n = distance_matrix(vecs_n)
D_n_df = pd.DataFrame(D_n, index=labels_n, columns=labels_n)
display(D_n_df)

C_n = compute_Ci(D_n, r)
C_n_df = pd.DataFrame({
    "Vector": labels_n,
    "Matches (d<=r, incl. self)": np.sum(D_n <= r, axis=1),
    "Ci": C_n
})
display(C_n_df)

phi_n = np.mean(np.log(C_n))
print(f"phi^(m+1) = {phi_n:.4f}")

Number of vectors = N - (m+1) + 1 = 3


,Vector,Values
0,Y1,"[4, 7, 9]"
1,Y2,"[7, 9, 10]"
2,Y3,"[9, 10, 6]"


,Y1,Y2,Y3
Y1,0.0,3.0,5.0
Y2,3.0,0.0,4.0
Y3,5.0,4.0,0.0


,Vector,"Matches (d<=r, incl. self)",Ci
0,Y1,2,0.666667
1,Y2,2,0.666667
2,Y3,1,0.333333


phi^(m+1) = -0.6365


### Step 7 — ApEn = phi^m − phi^(m+1)

In [15]:
ApEn_manual = phi_m - phi_n
print(f"ApEn = phi^m - phi^(m+1) = {phi_m:.4f} - ({phi_n:.4f}) = {ApEn_manual:.4f}")

ApEn = phi^m - phi^(m+1) = -0.5199 - (-0.6365) = 0.1167


---
# PART 2 — Sample Entropy (manual, worked step-by-step)

SampEn **excludes self-matches** and counts each pair only once (upper-triangle, i < j).

### Step 1 — Reuse length-m vectors, build pairwise (i<j) distance table, count matches B

In [16]:
def pairwise_matches(vecs, labels, r):
    rows = []
    n_vec = len(vecs)
    for i in range(n_vec):
        for j in range(i + 1, n_vec):
            d = np.max(np.abs(vecs[i] - vecs[j]))
            match = d <= r
            rows.append({"Pair": f"{labels[i]}-{labels[j]}", "Distance": d, f"Match (d<={r})": match})
    return pd.DataFrame(rows)

pairs_m = pairwise_matches(vecs_m, labels_m, r)
display(pairs_m)

B = pairs_m.iloc[:, 2].sum()
print(f"Number of matching pairs (length m={m}):  B = {B}")

,Pair,Distance,Match (d<=3)
0,x1-x2,3,True
1,x1-x3,5,False
2,x1-x4,6,False
3,x2-x3,2,True
4,x2-x4,3,True
5,x3-x4,4,False


Number of matching pairs (length m=2):  B = 3


### Step 2 — Same, for length m+1 = 3 vectors → count matches A

In [17]:
pairs_n = pairwise_matches(vecs_n, labels_n, r)
display(pairs_n)

A = pairs_n.iloc[:, 2].sum()
print(f"Number of matching pairs (length m+1={n}):  A = {A}")

,Pair,Distance,Match (d<=3)
0,Y1-Y2,3,True
1,Y1-Y3,5,False
2,Y2-Y3,4,False


Number of matching pairs (length m+1=3):  A = 1


### Step 3 — SampEn = -ln(A / B)

In [18]:
SampEn_manual = -np.log(A / B)
print(f"SampEn = -ln(A/B) = -ln({A}/{B}) = {SampEn_manual:.4f}")

SampEn = -ln(A/B) = -ln(1/3) = 1.0986


---
# PART 3 — Reusable functions (verified against Parts 1 & 2)

These are the same functions used earlier, now confirmed to reproduce the manual numbers exactly before we trust them on real data.

In [19]:
def _phi(series, m, r):
    vecs = form_vectors(series, m)
    D = distance_matrix(vecs)
    C = compute_Ci(D, r)
    return np.mean(np.log(C))


def approximate_entropy(series, m, r):
    series = np.asarray(series, dtype=float)
    return _phi(series, m, r) - _phi(series, m + 1, r)


def sample_entropy(series, m, r):
    series = np.asarray(series, dtype=float)

    def _count_matches(mlen):
        vecs = form_vectors(series, mlen)
        n_vec = len(vecs)
        count = 0
        for i in range(n_vec):
            for j in range(i + 1, n_vec):
                d = np.max(np.abs(vecs[i] - vecs[j]))
                if d <= r:
                    count += 1
        return count

    B = _count_matches(m)
    A = _count_matches(m + 1)

    if B == 0 or A == 0:
        return np.nan

    return -np.log(A / B)


# --- Verification against the manual worked example ---
check_apen = approximate_entropy(x, m, r)
check_sampen = sample_entropy(x, m, r)

print(f"Function ApEn   = {check_apen:.4f}   (manual = {ApEn_manual:.4f})")
print(f"Function SampEn = {check_sampen:.4f}   (manual = {SampEn_manual:.4f})")
assert np.isclose(check_apen, ApEn_manual), "ApEn mismatch!"
assert np.isclose(check_sampen, SampEn_manual), "SampEn mismatch!"
print("\nVerified: functions match the manual derivation exactly.")

Function ApEn   = 0.1167   (manual = 0.1167)
Function SampEn = 1.0986   (manual = 1.0986)

Verified: functions match the manual derivation exactly.


---
# PART 4 — Normalization of AF(d)  (Section 3.3 of the paper)

Before running entropy on real river data, the AF(d) series must be normalized:
- **d normalized by R** (max radial distance) → d/R runs 0–1
- **AF(d) normalized by total channel-intersection count** → AF/sum(AF), so it behaves like a probability distribution

This function will be applied per-year in Part 5.

In [20]:
def normalize_af(distances, af_vector):
    """
    distances : array of unique distances (meters), sorted ascending
    af_vector : array of channel counts at each distance
    Returns d_norm (d/R) and af_norm (AF/total)
    """
    distances = np.asarray(distances, dtype=float)
    af_vector = np.asarray(af_vector, dtype=float)

    R = distances.max()
    total = af_vector.sum()

    d_norm = distances / R
    af_norm = af_vector / total
    return d_norm, af_norm


# quick demo on a toy example
demo_d = np.array([5000, 10000, 15000, 20000])
demo_af = np.array([2, 3, 5, 1])
d_norm_demo, af_norm_demo = normalize_af(demo_d, demo_af)

pd.DataFrame({"d (m)": demo_d, "d_norm": d_norm_demo, "AF (raw)": demo_af, "AF_norm": af_norm_demo})

,d (m),d_norm,AF (raw),AF_norm
0,5000,0.25,2,0.181818
1,10000,0.50,3,0.272727
2,15000,0.75,5,0.454545
3,20000,1.00,1,0.090909


---
# PART 5 — Apply to real data (`Chennel_Data.xlsx`)

### Step A — Load file, list sheets (years)

In [21]:
FILE_PATH = r"G:\Cahnnel Complexity Ganges\Chennel_Data.xlsx"

xls = pd.ExcelFile(FILE_PATH)
sheet_names = xls.sheet_names
print("Sheets found (years):", sheet_names)

Sheets found (years): ['2010']


### Step B — Build raw AF vector per year (unique distance → count of channels)

In [22]:
def build_af_vector(file_path, sheet_name, distance_col="Distance"):
    df = pd.read_excel(file_path, sheet_name=sheet_name)
    counts = df[distance_col].value_counts().sort_index()
    distances = counts.index.to_numpy()
    af_vector = counts.to_numpy()
    return distances, af_vector


d_data = {}
af_data = {}

for sheet in sheet_names:
    distances, af_vector = build_af_vector(FILE_PATH, sheet)
    d_data[sheet] = distances
    af_data[sheet] = af_vector
    print(f"Year {sheet}: x_{sheet} (raw AF) = {af_vector}")

Year 2010: x_2010 (raw AF) = [1 2 2 2 1 1 2 2 3 4 3 3 3 3 3 2 4 2 2 3 5 4 5 1 2 3 2 3 2 3 3 1 2 2 1 2 9
 8 4 2 1]


### Step C — Normalize each year (d/R, AF/total)

In [23]:
d_norm_data = {}
af_norm_data = {}

for sheet in sheet_names:
    d_norm, af_norm = normalize_af(d_data[sheet], af_data[sheet])
    d_norm_data[sheet] = d_norm
    af_norm_data[sheet] = af_norm
    print(f"Year {sheet}: AF_norm = {np.round(af_norm, 4)}")

Year 2010: AF_norm = [0.0088 0.0177 0.0177 0.0177 0.0088 0.0088 0.0177 0.0177 0.0265 0.0354
 0.0265 0.0265 0.0265 0.0265 0.0265 0.0177 0.0354 0.0177 0.0177 0.0265
 0.0442 0.0354 0.0442 0.0088 0.0177 0.0265 0.0177 0.0265 0.0177 0.0265
 0.0265 0.0088 0.0177 0.0177 0.0088 0.0177 0.0796 0.0708 0.0354 0.0177
 0.0088]


### Step D — Compute r = 0.15 * std(AF_norm) per year, then ApEn and SampEn (m=2)

In [24]:
m_real = 2
results = []

for sheet in sheet_names:
    af_norm = af_norm_data[sheet]
    r_year = 0.15 * np.std(af_norm)

    apen_val = approximate_entropy(af_norm, m_real, r_year)
    sampen_val = sample_entropy(af_norm, m_real, r_year)

    results.append({
        "Year": sheet,
        "N": len(af_norm),
        "r": r_year,
        "ApEn": apen_val,
        "SampEn": sampen_val
    })

    print(f"Year {sheet}: r={r_year:.5f}, ApEn={apen_val:.4f}, SampEn={sampen_val:.4f}")

Year 2010: r=0.00221, ApEn=0.6676, SampEn=1.5476


### Step E — Results table

In [25]:
results_df = pd.DataFrame(results)
results_df

,Year,N,r,ApEn,SampEn
0,2010,41,0.002209,0.667649,1.547563


### Step F — Save results

In [ ]:
OUTPUT_PATH = r"G:\Cahnnel Complexity Ganges\AF_Entropy_Results.xlsx"
results_df.to_excel(OUTPUT_PATH, index=False)
print(f"Saved results to: {OUTPUT_PATH}")

### Step G — Plot ApEn and SampEn across years

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

x_pos = np.arange(len(results_df))
width = 0.4

ax1.bar(x_pos - width/2, results_df["ApEn"], width=width, label="ApEn", color="steelblue")
ax1.set_ylabel("Approximate Entropy (ApEn)", color="steelblue")
ax1.set_xticks(x_pos)
ax1.set_xticklabels(results_df["Year"], rotation=45)

ax2 = ax1.twinx()
ax2.bar(x_pos + width/2, results_df["SampEn"], width=width, label="SampEn", color="darkorange")
ax2.set_ylabel("Sample Entropy (SampEn)", color="darkorange")

ax1.set_xlabel("Year")
ax1.set_title("ApEn and SampEn of normalized AF series by Year")
fig.tight_layout()
plt.show()